In [ ]:
# individual_modeling_full_features.py
"""
Individual-level modeling using ALL available features from BRFSS
"""

import pandas as pd
import numpy as np
import os
import logging
from typing import List, Dict, Optional
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split
import warnings
warnings.filterwarnings('ignore')

# Set up logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

class IndividualLevelDataProcessorFull:
    """
    Process individual-level BRFSS data using ALL available features.
    """
    
    def __init__(self):
        self.brfss_individual = None
        self.epa_data = None
        self.merged_data = None
        self.features = None
        self.targets = None
        self.numeric_features = None
        self.categorical_features = None
        
    def load_individual_brfss(self, file_path: str = '../data/processed/brfss_individual_cleaned.csv'):
        """Load individual-level BRFSS data."""
        logger.info(f"Loading individual BRFSS data from {file_path}")
        
        if not os.path.exists(file_path):
            logger.error(f"File not found: {file_path}")
            raise FileNotFoundError(f"File not found: {file_path}")
        
        self.brfss_individual = pd.read_csv(file_path, low_memory=False)
        logger.info(f"Loaded {len(self.brfss_individual):,} individual records")
        logger.info(f"Total columns: {len(self.brfss_individual.columns)}")
        
        # Show column count by type
        numeric_cols = self.brfss_individual.select_dtypes(include=[np.number]).columns
        categorical_cols = self.brfss_individual.select_dtypes(include=['object', 'category']).columns
        
        logger.info(f"Numeric columns: {len(numeric_cols)}")
        logger.info(f"Categorical columns: {len(categorical_cols)}")
        
        return self.brfss_individual
    
    def load_epa_aqi_data(self, file_path: str = '../data/raw/annual_aqi_by_county_2023.csv'):
        """Load and process EPA AQI data."""
        logger.info(f"Loading EPA AQI data from {file_path}")
        
        if not os.path.exists(file_path):
            logger.warning(f"EPA file not found: {file_path}")
            return None
        
        # Process EPA data
        df = pd.read_csv(file_path)
        
        # Clean column names
        df.columns = [col.strip().replace('"', '') for col in df.columns]
        
        # Create state-level averages
        state_aqi_stats = df.groupby('State').agg({
            'Max AQI': 'mean',
            '90th Percentile AQI': 'mean',
            'Median AQI': 'mean',
            'Days with AQI': 'mean',
            'Good Days': 'mean',
            'Moderate Days': 'mean',
            'Unhealthy for Sensitive Groups Days': 'mean'
        }).reset_index()
        
        state_aqi_stats.columns = [
            'State', 'State_Max_AQI', 'State_90th_AQI', 'State_Median_AQI',
            'State_AQI_Days', 'State_Good_Days', 'State_Moderate_Days',
            'State_Unhealthy_Sensitive_Days'
        ]
        
        # Calculate percentages
        state_aqi_stats['State_Good_Days_Pct'] = (state_aqi_stats['State_Good_Days'] / state_aqi_stats['State_AQI_Days']) * 100
        state_aqi_stats['State_Unhealthy_Days_Pct'] = (
            (state_aqi_stats['State_Unhealthy_Sensitive_Days']) / state_aqi_stats['State_AQI_Days']
        ) * 100
        
        self.epa_data = state_aqi_stats
        logger.info(f"Processed EPA data for {len(self.epa_data)} states")
        
        return self.epa_data
    
    def merge_data(self):
        """Merge individual BRFSS with state-level EPA data."""
        logger.info("Merging individual BRFSS with EPA data...")
        
        if self.brfss_individual is None:
            logger.error("BRFSS data not loaded. Call load_individual_brfss() first.")
            return None
        
        # If EPA data is not available, use only BRFSS
        if self.epa_data is None:
            logger.warning("EPA data not available. Using BRFSS data only.")
            self.merged_data = self.brfss_individual.copy()
            return self.merged_data
        
        # Merge on State
        self.merged_data = self.brfss_individual.merge(
            self.epa_data, 
            on='State', 
            how='left'
        )
        
        logger.info(f"Merged dataset: {len(self.merged_data):,} individuals")
        logger.info(f"Columns after merge: {len(self.merged_data.columns)}")
        
        return self.merged_data
    
    def select_features_automatically(self, missing_threshold: float = 0.3):
        """
        Automatically select features based on data quality.
        
        Args:
            missing_threshold: Maximum allowed missing value percentage (0-1)
        """
        logger.info(f"Automatically selecting features (missing threshold: {missing_threshold*100}%)...")
        
        if self.merged_data is None:
            logger.error("No merged data available. Call merge_data() first.")
            return None, None
        
        # Columns to always exclude
        always_exclude = [
            'FIPS', 'State', 'Sample_Size', 'County_Cluster',
            'State_FIPS', 'County_FIPS'
        ]
        
        # Target columns (we'll use these separately)
        target_candidates = [
            'Diabetes', 'HeartDisease', 'Stroke', 'Asthma',
            'HeartAttack', 'COPD', 'Depression'
        ]
        
        # Initialize feature list
        candidate_features = []
        
        for col in self.merged_data.columns:
            # Skip always excluded columns
            if col in always_exclude:
                continue
                
            # Skip target columns (we'll handle separately)
            if col in target_candidates:
                continue
                
            # Skip if too many missing values
            missing_pct = self.merged_data[col].isnull().mean()
            if missing_pct > missing_threshold:
                logger.debug(f"Excluding {col}: {missing_pct:.1%} missing (>{missing_threshold:.0%})")
                continue
            
            # Skip constant columns
            if self.merged_data[col].nunique() <= 1:
                logger.debug(f"Excluding {col}: constant column")
                continue
                
            candidate_features.append(col)
        
        # Separate numeric and categorical features
        self.numeric_features = [
            col for col in candidate_features 
            if pd.api.types.is_numeric_dtype(self.merged_data[col])
        ]
        
        self.categorical_features = [
            col for col in candidate_features 
            if col not in self.numeric_features
        ]
        
        self.features = candidate_features
        
        # Select targets that exist and have reasonable data
        self.targets = []
        for target in ['Diabetes', 'HeartDisease', 'Stroke', 'Asthma']:
            if target in self.merged_data.columns:
                missing_pct = self.merged_data[target].isnull().mean()
                if missing_pct <= missing_threshold:
                    self.targets.append(target)
                    logger.info(f"  Target {target}: {self.merged_data[target].mean()*100:.1f}% prevalence")
                else:
                    logger.warning(f"  Target {target} has {missing_pct:.1%} missing values, excluding")
        
        logger.info(f"Selected {len(self.features)} features:")
        logger.info(f"  - Numeric: {len(self.numeric_features)}")
        logger.info(f"  - Categorical: {len(self.categorical_features)}")
        logger.info(f"Selected {len(self.targets)} targets: {self.targets}")
        
        return self.features, self.targets
    
    def prepare_modeling_data(self, 
                            sample_size: Optional[int] = None,
                            test_size: float = 0.2,
                            random_state: int = 42):
        """
        Prepare data for modeling with preprocessing pipeline.
        
        Returns:
            Dictionary with preprocessed data
        """
        logger.info("Preparing modeling data with preprocessing pipeline...")
        
        if self.merged_data is None or self.features is None:
            logger.error("Data not prepared. Call merge_data() and select_features() first.")
            return {}
        
        # Sample if requested
        if sample_size and sample_size < len(self.merged_data):
            df = self.merged_data.sample(n=sample_size, random_state=random_state)
            logger.info(f"Using sample of {sample_size:,} individuals")
        else:
            df = self.merged_data.copy()
            logger.info(f"Using all {len(df):,} individuals")
        
        # Prepare X and y
        X = df[self.features].copy()
        y = df[self.targets].copy()
        
        # Fill NaN in y with 0 (assuming NaN = no disease)
        y = y.fillna(0)
        
        logger.info(f"Initial X shape: {X.shape}")
        logger.info(f"Initial y shape: {y.shape}")
        
        # Create preprocessing pipeline
        numeric_transformer = Pipeline(steps=[
            ('imputer', SimpleImputer(strategy='median')),
            ('scaler', StandardScaler())
        ])
        
        categorical_transformer = Pipeline(steps=[
            ('imputer', SimpleImputer(strategy='most_frequent')),
            ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
        ])
        
        preprocessor = ColumnTransformer(
            transformers=[
                ('num', numeric_transformer, self.numeric_features),
                ('cat', categorical_transformer, self.categorical_features)
            ])
        
        # Fit and transform
        logger.info("Fitting preprocessing pipeline...")
        X_processed = preprocessor.fit_transform(X)
        
        # Get feature names after one-hot encoding
        if hasattr(preprocessor.named_transformers_['cat']['onehot'], 'get_feature_names_out'):
            cat_features = preprocessor.named_transformers_['cat']['onehot'].get_feature_names_out(self.categorical_features)
        else:
            cat_features = []
            
        all_feature_names = list(self.numeric_features) + list(cat_features)
        
        logger.info(f"Processed X shape: {X_processed.shape}")
        logger.info(f"Number of features after encoding: {len(all_feature_names)}")
        
        # Train-test split
        X_train, X_test, y_train, y_test = train_test_split(
            X_processed, y, test_size=test_size, random_state=random_state,
            stratify=y[self.targets[0]] if len(self.targets) > 0 else None
        )
        
        logger.info(f"Train set: {X_train.shape}")
        logger.info(f"Test set: {X_test.shape}")
        
        return {
            'X_train': X_train,
            'X_test': X_test,
            'y_train': y_train.values,
            'y_test': y_test.values,
            'feature_names': all_feature_names,
            'target_names': self.targets,
            'preprocessor': preprocessor,
            'X_train_raw': X.values,
            'X_test_raw': X_test
        }


def main():
    """Main execution function."""
    logger.info("=" * 70)
    logger.info("INDIVIDUAL-LEVEL MODELING WITH FULL FEATURES")
    logger.info("=" * 70)
    
    # Create directories
    os.makedirs('data/processed/individual_full', exist_ok=True)
    os.makedirs('reports/individual_full', exist_ok=True)
    
    try:
        # Initialize processor
        processor = IndividualLevelDataProcessorFull()
        
        # Step 1: Load individual BRFSS data
        logger.info("\n[STEP 1] Loading individual BRFSS data...")
        brfss_data = processor.load_individual_brfss()
        
        # Step 2: Load EPA data
        logger.info("\n[STEP 2] Loading EPA AQI data...")
        epa_data = processor.load_epa_aqi_data()
        
        # Step 3: Merge data
        logger.info("\n[STEP 3] Merging datasets...")
        merged_data = processor.merge_data()
        
        # Step 4: Automatically select features
        logger.info("\n[STEP 4] Selecting features automatically...")
        features, targets = processor.select_features_automatically(missing_threshold=0.3)
        
        # Step 5: Save dataset
        logger.info("\n[STEP 5] Saving dataset...")
        merged_data.to_csv('data/processed/individual_full/full_dataset.csv', index=False)
        
        # Step 6: Prepare modeling data (use all data)
        logger.info("\n[STEP 6] Preparing modeling data...")
        modeling_data = processor.prepare_modeling_data(
            sample_size=None,  # Use ALL data
            test_size=0.2,
            random_state=42
        )
        
        # Save modeling data
        import joblib
        joblib.dump(modeling_data, 'data/processed/individual_full/modeling_data.pkl')
        logger.info("Saved modeling data to data/processed/individual_full/modeling_data.pkl")
        
        # Save preprocessor
        joblib.dump(modeling_data['preprocessor'], 'data/processed/individual_full/preprocessor.pkl')
        
        # Generate summary
        logger.info("\n" + "=" * 70)
        logger.info("PIPELINE COMPLETE - SUMMARY")
        logger.info("=" * 70)
        
        logger.info(f"Total individuals: {len(merged_data):,}")
        logger.info(f"Original features selected: {len(features)}")
        logger.info(f"Features after encoding: {len(modeling_data['feature_names'])}")
        logger.info(f"Targets: {len(targets)}")
        logger.info(f"Train samples: {modeling_data['X_train'].shape[0]:,}")
        logger.info(f"Test samples: {modeling_data['X_test'].shape[0]:,}")
        
        # Show some feature examples
        logger.info(f"\nSample features (first 20):")
        for i, feat in enumerate(modeling_data['feature_names'][:20]):
            logger.info(f"  {i+1}. {feat}")
        
        return processor, modeling_data
        
    except Exception as e:
        logger.error(f"Pipeline failed: {e}")
        import traceback
        traceback.print_exc()
        return None


if __name__ == "__main__":
    processor, modeling_data = main()

2025-12-14 16:27:02,116 - INFO - ======================================================================
2025-12-14 16:27:02,117 - INFO - INDIVIDUAL-LEVEL MODELING WITH FULL FEATURES
2025-12-14 16:27:02,118 - INFO - ======================================================================
2025-12-14 16:27:02,118 - INFO - 
[STEP 1] Loading individual BRFSS data...
2025-12-14 16:27:02,118 - INFO - Loading individual BRFSS data from ../data/processed/brfss_individual_cleaned.csv


2025-12-14 16:27:05,270 - INFO - Loaded 433,323 individual records
2025-12-14 16:27:05,271 - INFO - Total columns: 70
2025-12-14 16:27:05,357 - INFO - Numeric columns: 66
2025-12-14 16:27:05,358 - INFO - Categorical columns: 4
2025-12-14 16:27:05,358 - INFO - 
[STEP 2] Loading EPA AQI data...
2025-12-14 16:27:05,358 - INFO - Loading EPA AQI data from ../data/raw/annual_aqi_by_county_2023.csv
2025-12-14 16:27:05,369 - INFO - Processed EPA data for 54 states
2025-12-14 16:27:05,370 - INFO - 
[STEP 3] Merging datasets...
2025-12-14 16:27:05,370 - INFO - Merging individual BRFSS with EPA data...
2025-12-14 16:27:05,484 - INFO - Merged dataset: 433,323 individuals
2025-12-14 16:27:05,485 - INFO - Columns after merge: 79
2025-12-14 16:27:05,485 - INFO - 
[STEP 4] Selecting features automatically...
2025-12-14 16:27:05,485 - INFO - Automatically selecting features (missing threshold: 30.0%)...
2025-12-14 16:27:05,837 - INFO -   Target Diabetes: 13.8% prevalence
2025-12-14 16:27:05,842 - INFO 

In [3]:
# check_all_columns.py
import pandas as pd
import numpy as np

# Load your individual data
individual_path = '../data/processed/brfss_individual_cleaned.csv'
print(f"Loading individual data from: {individual_path}")
df = pd.read_csv(individual_path, nrows=1000)  # Load first 1000 rows to check

print(f"\nDataset shape: {df.shape}")
print(f"Total columns: {len(df.columns)}")

# Categorize columns
print("\n" + "="*50)
print("COLUMN CATEGORIZATION")
print("="*50)

# Group columns by type/purpose
column_categories = {
    'Demographics': [],
    'Clinical Measurements': [],
    'Lifestyle/Behavioral': [],
    'Diseases/Conditions': [],
    'Healthcare Access': [],
    'Environmental': [],
    'Composite Scores': [],
    'Other': []
}

for col in df.columns:
    col_lower = col.lower()
    
    # Categorize based on column name patterns
    if any(word in col_lower for word in ['age', 'sex', 'race', 'educ', 'income', 'marital', 'employ', 'veteran', 'child']):
        column_categories['Demographics'].append(col)
    elif any(word in col_lower for word in ['bmi', 'height', 'weight', 'bp', 'chol', 'blood', 'glucose', 'pressure']):
        column_categories['Clinical Measurements'].append(col)
    elif any(word in col_lower for word in ['smok', 'drink', 'alcohol', 'physical', 'fruit', 'veg', 'sleep', 'sedentary', 'exercise', 'salt']):
        column_categories['Lifestyle/Behavioral'].append(col)
    elif any(word in col_lower for word in ['diabetes', 'heart', 'stroke', 'asthma', 'cancer', 'copd', 'depress', 'arthrit', 'kidney']):
        column_categories['Diseases/Conditions'].append(col)
    elif any(word in col_lower for word in ['healthcare', 'checkup', 'dental', 'flu', 'hiv', 'access', 'coverage']):
        column_categories['Healthcare Access'].append(col)
    elif any(word in col_lower for word in ['aqi', 'pollut', 'air', 'environment']):
        column_categories['Environmental'].append(col)
    elif any(word in col_lower for word in ['score', 'index', 'composite', 'risk', 'severity']):
        column_categories['Composite Scores'].append(col)
    else:
        column_categories['Other'].append(col)

# Print categorization
for category, cols in column_categories.items():
    if cols:
        print(f"\n{category} ({len(cols)} columns):")
        for col in cols[:10]:  # Show first 10
            print(f"  - {col}")
        if len(cols) > 10:
            print(f"  ... and {len(cols) - 10} more")

# Show data types and missing values
print("\n" + "="*50)
print("DATA TYPE & MISSING VALUE ANALYSIS")
print("="*50)

dtype_counts = df.dtypes.value_counts()
print("\nData types:")
for dtype, count in dtype_counts.items():
    print(f"  {dtype}: {count} columns")

print("\nMissing value analysis (top 20 columns with most missing):")
missing_pct = (df.isnull().sum() / len(df) * 100).sort_values(ascending=False)
for col, pct in missing_pct.head(20).items():
    print(f"  {col}: {pct:.1f}% missing")

print("\n" + "="*50)
print("RECOMMENDED FEATURES FOR MODELING")
print("="*50)

# Recommend features based on low missing values and relevance
recommended_features = []
for col in df.columns:
    # Skip if too many missing values (>30%)
    if df[col].isnull().mean() > 0.3:
        continue
    
    # Skip identifiers and targets (we'll add targets separately)
    if col in ['FIPS', 'State', 'Sample_Size', 'County_Cluster']:
        continue
    
    # Skip if not a good predictor type
    if df[col].nunique() == 1:  # Constant column
        continue
        
    recommended_features.append(col)

print(f"Recommended features (based on <30% missing): {len(recommended_features)}")
print("\nSample of recommended features:")
for col in recommended_features[:20]:
    unique_vals = df[col].nunique()
    dtype = df[col].dtype
    print(f"  {col} (dtype: {dtype}, unique: {unique_vals})")

Loading individual data from: ../data/processed/brfss_individual_cleaned.csv

Dataset shape: (1000, 70)
Total columns: 70

COLUMN CATEGORIZATION

Demographics (10 columns):
  - Age
  - Sex
  - Race
  - Education
  - Income
  - MaritalStatus
  - Employment
  - Veteran
  - Children
  - Age_Group

Clinical Measurements (7 columns):
  - BMI
  - Height
  - Weight
  - HighBP
  - HighChol
  - CholCheck
  - BMI_Category

Lifestyle/Behavioral (13 columns):
  - Smoker
  - SmokeStatus
  - HeavyDrinker
  - AlcoholConsumption
  - PhysicalActivity
  - FruitCons
  - VegCons
  - SleepHours
  - SedentaryTime
  - PhysicalHealthDays
  ... and 3 more

Diseases/Conditions (12 columns):
  - PainArthritis
  - HeartDisease
  - HeartAttack
  - Stroke
  - Asthma
  - SkinCancer
  - OtherCancer
  - COPD
  - Depression
  - KidneyDisease
  ... and 2 more

Healthcare Access (6 columns):
  - AnyHealthcare
  - Checkup
  - DentalVisit
  - FluShot
  - HIVTest
  - HealthcareAccess_Score

Composite Scores (5 columns):
  -